In [1]:
# ============================================
# QC002 - La decoherencia cuántica nunca se vio así
# Datos reales de IBM Quantum
# ============================================
import numpy as np
import pandas as pd
import qutip as qt
import os

export_path = "/home/vfx/Documents/QuantumCanvas/science/exports/"
os.makedirs(export_path, exist_ok=True)

# Parametros reales de IBM Quantum (2024)
# T1 = tiempo de relajacion
# T2 = tiempo de desfase
T1 = 150e-6   # 150 microsegundos
T2 = 80e-6    # 80 microsegundos
frames = 120

# Tasas de decoherencia
gamma1 = 1/T1   # relajacion
gamma2 = 1/T2   # desfase

# Tiempo total de simulacion = 3*T2
t_total = 3 * T2
times = np.linspace(0, t_total, frames)

# Estado inicial: superposicion perfecta |+>
psi0 = (qt.basis(2,0) + qt.basis(2,1)).unit()

# Operadores de colapso (Lindblad)
c_ops = [
    np.sqrt(gamma1) * qt.destroy(2),  # relajacion T1
    np.sqrt(gamma2/2) * qt.sigmaz(),  # desfase T2
]

# Resolver ecuacion maestra de Lindblad
result = qt.mesolve(
    qt.sigmax(),   # Hamiltoniano
    psi0,
    times,
    c_ops,
    [qt.sigmax(), qt.sigmay(), qt.sigmaz()]
)

# Extraer valores esperados (coordenadas Bloch)
rows = []
for i, t in enumerate(times):
    x = result.expect[0][i]
    y = result.expect[1][i]
    z = result.expect[2][i]

    # Pureza desde la longitud del vector de Bloch
    purity = np.sqrt(x**2 + y**2 + z**2)

    rows.append({
        "frame":         i + 1,
        "houdini_frame": i + 1,
        "x":             x,
        "y":             y,
        "z":             z,
        "purity":        purity,
        "time_us":       t * 1e6,  # en microsegundos
        "T1_us":         T1 * 1e6,
        "T2_us":         T2 * 1e6,
    })

df = pd.DataFrame(rows)
output = export_path + "QC002_decoherence_real.csv"
df.to_csv(output, index=False, float_format="%.8f")

print(f"✅ Exportado: {output}")
print(f"   T1 = {T1*1e6:.0f} us (IBM Quantum real)")
print(f"   T2 = {T2*1e6:.0f} us (IBM Quantum real)")
print(f"   Frames: {len(df)}")
print(f"\nInicio → purity={df.iloc[0].purity:.4f}  z={df.iloc[0].z:.4f}")
print(f"Mitad  → purity={df.iloc[59].purity:.4f}  z={df.iloc[59].z:.4f}")
print(f"Final  → purity={df.iloc[-1].purity:.4f}  z={df.iloc[-1].z:.4f}")

✅ Exportado: /home/vfx/Documents/QuantumCanvas/science/exports/QC002_decoherence_real.csv
   T1 = 150 us (IBM Quantum real)
   T2 = 80 us (IBM Quantum real)
   Frames: 120

Inicio → purity=1.0000  z=0.0000
Mitad  → purity=0.5683  z=0.5476
Final  → purity=0.7984  z=0.7981


/home/vfx/miniconda3/envs/quantumcanvas/lib/python3.11/site-packages/qutip/solver/solver_base.py:598: FutureWarning: e_ops will be keyword only from qutip 5.3 for all solver
  warnings.warn(


In [ ]:
no es todo
